# Graph + FGW with human-labeled examples

Each tagged transcript is a graph (one node per cue, plus the start node). One FGW barycenter per level is computed from its labeled examples (as many nodes as the median of their graphs), and every transcript gets the level of the nearest barycenter.

**Data**: a csv with one row per video transcript: its tagged text (`Transcript`), the level given by humans (`ability_label_name`), the number of human ratings (`ability_n`) and, optionally, the reference level instructed to the LLM (`REFERENCE_COLUMN`), levels being Low / Medium / High.

**Protocol** (identical in `human_fgw.ipynb` and `human_embeddings.ipynb`):
* **examples**: the videos rated by more than 4 humans (`EXAMPLES_QUERY = "ability_n > 4"`), whose labels are the most reliable; all of them (`M = None`) or $m$ per level drawn among them;
* **evaluation**: all the other videos;
* the examples are labeled either by the **humans** or by the **reference** level, and every classification is scored against **both** labels: accuracy, macro-F1, quadratic-weighted Cohen's $\kappa$ with a 95% bootstrap CI over the evaluated videos;
* the examples are saved in `splits/` and reloaded, so both methods use exactly the same ones;
* **cross-validation over the example videos**: they are split into `N_FOLDS` stratified folds; each fold in turn is left out and the barycenters are made from the other folds (or `M` per level drawn from them), always evaluated on the same other videos (mean ± SD over folds): how much the results depend on which reliable videos are used;
* **unsupervised**: $k$-means with $K = 3$ on the evaluated videos, without any label; each cluster is mapped to a level with the Hungarian algorithm (separately for each label evaluated against). ARI and AMI do not depend on that mapping. Repeated with `N_REPEATS` initializations.

In [ ]:
import os
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, kappa_with_ci, load_labeled_transcripts, load_or_create_examples,
                        load_or_create_pool_examples, load_or_create_split, map_clusters, plot_confusion)
from graph_multimodalsentence import Graph, compute_barycenter
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
DIMENSION = "ability"           # names the splits and outputs

EXAMPLES_QUERY = "ability_n > 4"  # videos used as examples (pandas query on the columns of the csv);
                                 # all the other videos are evaluated
SPLIT_DIR = "splits"            # shared with the other notebook
OUTPUT_DIR = "results"

M = None                        # examples per level drawn among them (None: all of them)
N_FOLDS = 5                     # cross-validation over the example videos
SEED = 0                        # folds, draw of the examples (with M) and k-means initialization
N_REPEATS = 5                   # k-means initializations

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
BARYCENTER_MAX_ITER = 50
N_KMEANS_INIT = 5               # k-means restarts (unsupervised)
KMEANS_MAX_ITER = 20            # k-means iterations per restart

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Labeled videos

In [ ]:
if REFERENCE_COLUMN is None:
    columns = pd.read_csv(LABELS_FILE, nrows=0).columns
    REFERENCE_COLUMN = next((c for c in columns if c not in (TEXT_COLUMN, LABEL_COLUMN)
                             and re.search("ref|instruct", c, re.IGNORECASE)), None)
print("reference column:", REFERENCE_COLUMN)

df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN)
LABELS = {"human": "level", **({"reference": "reference"} if REFERENCE_COLUMN else {})}  # label -> column of df

# example videos: rows of the csv selected by EXAMPLES_QUERY (ids are row<k>)
example_rows = pd.read_csv(LABELS_FILE).query(EXAMPLES_QUERY).index
df["role"] = np.where(df["id"].isin([f"row{k}" for k in example_rows]), "examples", "evaluated")
pool = df[df["role"] == "examples"]
test = df[df["role"] == "evaluated"]
print(f"{len(pool)} example videos ({EXAMPLES_QUERY}), {len(test)} evaluated videos")
pd.concat({role: pd.DataFrame({label: part[column].value_counts() for label, column in LABELS.items()})
           for role, part in df.groupby("role")}, axis=1).reindex(LEVELS).fillna(0).astype(int)

### Agreement between the human and the reference levels

How often humans recognize the instructed level, on the evaluated videos: the context for the scores below.

In [ ]:
if "reference" in LABELS:
    agreement = kappa_with_ci(pd.DataFrame({"true": test["reference"], "pred": test["level"]}))
    print(f"exact agreement: {(test['reference'] == test['level']).mean():.3f}, "
          f"kappa = {agreement['kappa']:.2f} [{agreement['kappa_ci_low']:.2f}, {agreement['kappa_ci_high']:.2f}]")
    display(pd.crosstab(test["reference"], test["level"], rownames=["reference"], colnames=["human"])
            .reindex(index=LEVELS, columns=LEVELS, fill_value=0))

## Graphs

Cue descriptions are embedded once (the same descriptions occur in many transcripts).

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))
graphs = {i: Graph.from_sentence(text, embedding) for i, text in zip(df["id"], df["text"])}
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")

## Nearest FGW barycenter

In [ ]:
def classify(folds, examples):
    # Predicted level of every transcript, from the barycenters of its own fold.
    predictions = []
    for fold in sorted(folds["fold"].unique()):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        barycenters = [compute_barycenter([graphs[i] for i in support.loc[support["level"] == level, "id"]],
                                          alpha=ALPHA, max_iter=BARYCENTER_MAX_ITER) for level in LEVELS]
        distances = np.array([[fgw.graph_d(graphs[i], b) for b in barycenters] for i in test["id"]])
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
    return pd.concat(predictions, ignore_index=True)


def cluster(ids, seed):
    # FGW k-means (K = 3) on the transcripts of ids: centroids are FGW barycenters; each restart
    # starts from K transcripts drawn at random, and the restart with the lowest inertia is kept.
    k = len(LEVELS)
    rng = np.random.default_rng(seed)
    best_labels, best_inertia = None, np.inf
    for restart in range(N_KMEANS_INIT):
        centers = [graphs[i] for i in rng.choice(ids, k, replace=False)]
        labels = None
        for iteration in range(KMEANS_MAX_ITER):
            distances = np.array([[fgw.graph_d(graphs[i], c) for c in centers] for i in ids])
            new_labels = distances.argmin(axis=1)
            if labels is not None and np.array_equal(new_labels, labels):
                break
            labels = new_labels
            for c in range(k):
                members = [i for i, label in zip(ids, labels) if label == c]
                if not members:  # empty cluster: restart it from the graph farthest from its center
                    members = [ids[distances.min(axis=1).argmax()]]
                centers[c] = compute_barycenter([graphs[i] for i in members], alpha=ALPHA,
                                                max_iter=BARYCENTER_MAX_ITER)
        distances = np.array([[fgw.graph_d(graphs[i], c) for c in centers] for i in ids])
        labels = distances.argmin(axis=1)
        inertia = distances[np.arange(len(ids)), labels].sum()
        if inertia < best_inertia:
            best_labels, best_inertia = labels, inertia
    return pd.DataFrame({"id": ids, "cluster": best_labels})

## Classification

Examples labeled by humans or by the reference level; every classification of the evaluated videos is scored against both labels.

In [ ]:
METHOD = "graph + FGW"
TAG = f"{DIMENSION}_{re.sub(r'[^A-Za-z0-9]+', '', EXAMPLES_QUERY)}"  # names the saved examples
folds = test[["id", "level"]].assign(fold=0)  # a single "fold": the evaluated videos


def run(seed):
    # Predictions with examples labeled by each label (drawn with this seed when M is set).
    predictions = {}
    for source, column in LABELS.items():
        examples = load_or_create_pool_examples(pool["id"], pool[column], f"{TAG}_{source}_seed{seed}",
                                                SPLIT_DIR, M, seed)
        pred = classify(folds, examples).drop(columns=["true", "fold"])
        predictions[source] = pred.merge(df[["id", *LABELS.values()]], on="id")
    return predictions


def score_all(predictions):
    # Scores of every classification against every label.
    results = []
    for source, pred in predictions.items():
        for target, column in LABELS.items():
            results.append(evaluate({(DIMENSION, f"examples: {source}"): pred.assign(true=pred[column])},
                                    method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


predictions = run(SEED)
for source, pred in predictions.items():
    pred.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_predictions_{DIMENSION}_{source}_examples.csv"), index=False)
results = score_all(predictions).drop(columns="fold")
results.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_results_{DIMENSION}.csv"), index=False)
results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                             zip(results["kappa"], results["kappa_ci_low"], results["kappa_ci_high"])]
results.pivot(index="examples", columns="evaluated against", values=["accuracy", "macro_f1", "kappa [95% CI]"])

Confusion matrices (rows: label evaluated against; columns: examples used).

In [ ]:
plot_confusion({(f"vs {target}", f"examples: {source}"): pred.assign(true=pred[column])
                for target, column in LABELS.items() for source, pred in predictions.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_fgw_confusion_{DIMENSION}.png"))
plt.show()

## Cross-validation over the example videos

The example videos are split into `N_FOLDS` folds (stratified on the human level). Each fold in turn is left out and the examples come from the other folds (all of them, or `M` per level drawn among them); the evaluated videos are always the same. Mean ± SD over folds.

In [ ]:
pool_folds, pool_human_examples = load_or_create_split(
    pool[["id", "level"]], f"{TAG}_pool", "human", SPLIT_DIR, M, N_FOLDS, SEED)
cv_examples = {"human": pool_human_examples}
for source, column in LABELS.items():
    if source != "human":
        cv_examples[source] = load_or_create_examples(
            pool_folds, pool.set_index("id").loc[pool_folds["id"], column], f"{TAG}_pool_{source}",
            SPLIT_DIR, M, SEED)

cv = []
for fold in range(N_FOLDS):
    fold_predictions = {}
    for source, examples in cv_examples.items():
        pred = classify(folds, examples[examples["fold"] == fold].assign(fold=0)).drop(columns=["true", "fold"])
        fold_predictions[source] = pred.merge(df[["id", *LABELS.values()]], on="id")
    cv.append(score_all(fold_predictions).query("fold == 'all'").assign(fold=fold))
cv = pd.concat(cv, ignore_index=True)[["fold", "method", "examples", "evaluated against", "accuracy", "macro_f1", "kappa"]]
cv.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_cv_{DIMENSION}.csv"), index=False)
cv.groupby(["examples", "evaluated against"])[["accuracy", "macro_f1", "kappa"]].agg(["mean", "std"]).round(3)

## Unsupervised: $k$-means and Hungarian mapping

On the evaluated videos (the same as above), without any label; the labels only map clusters to levels for scoring.

In [ ]:
UNSUPERVISED = "none (k-means)"


def run_unsupervised(seed):
    return cluster(list(test["id"]), seed).merge(df[["id", *LABELS.values()]], on="id")


def score_unsupervised(pred):
    # Scores against every label, each with its own cluster-to-level mapping.
    results = []
    for target, column in LABELS.items():
        mapped = pred.assign(true=pred[column], pred=map_clusters(pred["cluster"], pred[column]))
        results.append(evaluate({(DIMENSION, UNSUPERVISED): mapped}, method=METHOD, reference=target))
    return pd.concat(results, ignore_index=True).rename(columns={"setting": "examples", "reference": "evaluated against"})


unsupervised = run_unsupervised(SEED)
unsupervised.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_unsupervised_predictions_{DIMENSION}.csv"), index=False)
unsupervised_results = score_unsupervised(unsupervised).drop(columns="fold")
unsupervised_results.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_unsupervised_results_{DIMENSION}.csv"), index=False)
unsupervised_results["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in zip(
    unsupervised_results["kappa"], unsupervised_results["kappa_ci_low"], unsupervised_results["kappa_ci_high"])]
unsupervised_results[["evaluated against", "accuracy", "macro_f1", "kappa [95% CI]", "ARI", "AMI"]].round(3)

In [ ]:
plot_confusion({(f"vs {target}", UNSUPERVISED):
                unsupervised.assign(true=unsupervised[column], pred=map_clusters(unsupervised["cluster"], unsupervised[column]))
                for target, column in LABELS.items()},
               "true", os.path.join(OUTPUT_DIR, f"human_fgw_unsupervised_confusion_{DIMENSION}.png"))
plt.show()

Over `N_REPEATS` initializations of $k$-means:

In [ ]:
unsupervised_repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    seed_results = score_unsupervised(unsupervised if seed == SEED else run_unsupervised(seed))
    unsupervised_repeats.append(seed_results.assign(seed=seed))
unsupervised_repeats = pd.concat(unsupervised_repeats, ignore_index=True)[
    ["seed", "method", "examples", "evaluated against", "accuracy", "macro_f1", "kappa", "ARI", "AMI"]]
unsupervised_repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_unsupervised_repeats_{DIMENSION}.csv"), index=False)
unsupervised_repeats.groupby("evaluated against")[["accuracy", "macro_f1", "kappa", "ARI", "AMI"]].agg(["mean", "std"]).round(3)

## Comparison with the embedding baseline

If `human_embeddings.ipynb` was run with the same parameters (same examples and evaluated videos).

In [ ]:
paths = {"results": os.path.join(OUTPUT_DIR, f"human_embedding_results_{DIMENSION}.csv"),
         "unsupervised": os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_results_{DIMENSION}.csv"),
         "cv": os.path.join(OUTPUT_DIR, f"human_embedding_cv_{DIMENSION}.csv"),
         "unsupervised_repeats": os.path.join(OUTPUT_DIR, f"human_embedding_unsupervised_repeats_{DIMENSION}.csv")}
if os.path.exists(paths["results"]) and os.path.exists(paths["unsupervised"]):
    both = pd.concat([results, unsupervised_results, pd.read_csv(paths["results"]), pd.read_csv(paths["unsupervised"])])
    display(both.pivot_table(index=["examples", "evaluated against"], columns="method",
                             values=["accuracy", "macro_f1", "kappa"]).round(3))
if os.path.exists(paths["cv"]) and os.path.exists(paths["unsupervised_repeats"]):
    # supervised: mean and SD over the cross-validation folds; unsupervised: over k-means initializations
    display(pd.concat([cv, unsupervised_repeats, pd.read_csv(paths["cv"]),
                       pd.read_csv(paths["unsupervised_repeats"])])
            .groupby(["examples", "evaluated against", "method"])[["accuracy", "macro_f1", "kappa"]]
            .agg(["mean", "std"]).round(3))